# 01 — Exploratory Data Analysis: Clinical Biomarkers

This notebook explores the clinical/tabular dataset used for biomarker-based DR risk prediction.

**Goals:**
- Understand data distributions, missing values, and class imbalance
- Visualize feature correlations with DR severity
- Identify the most predictive biomarkers

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import load_settings

settings = load_settings()
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Settings loaded successfully.')

## 1. Load the Clinical Dataset

In [ ]:
# Load raw clinical data
csv_path = settings['paths']['raw_tabular']
df = pd.read_csv(csv_path)

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 2. Basic Statistics & Missing Values

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'percent': missing_pct})
missing_df = missing_df[missing_df['count'] > 0].sort_values('count', ascending=False)

if len(missing_df) > 0:
    print('Missing values:')
    display(missing_df)
    
    fig, ax = plt.subplots(figsize=(10, 4))
    missing_df['percent'].plot(kind='barh', ax=ax, color='coral')
    ax.set_xlabel('Missing %')
    ax.set_title('Missing Values by Feature')
    plt.tight_layout()
    plt.show()
else:
    print('No missing values found.')

## 3. Target Distribution (DR Grade)

In [ ]:
target = settings['tabular']['target']
grade_labels = {0: 'No DR', 1: 'Mild', 2: 'Moderate', 3: 'Severe', 4: 'Proliferative'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
counts = df[target].value_counts().sort_index()
colors = sns.color_palette('RdYlGn_r', n_colors=5)
axes[0].bar(counts.index, counts.values, color=colors)
axes[0].set_xticks(range(5))
axes[0].set_xticklabels([grade_labels[i] for i in range(5)], rotation=15)
axes[0].set_title('DR Grade Distribution')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(counts.values, labels=[grade_labels[i] for i in counts.index],
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('DR Grade Proportions')

plt.tight_layout()
plt.show()

print(f'Class counts:\n{counts}')

## 4. Feature Distributions by DR Grade

In [ ]:
features = settings['tabular']['features']
numeric_features = [f for f in features if f in df.select_dtypes(include=[np.number]).columns]

n_cols = 3
n_rows = (len(numeric_features) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, feat in enumerate(numeric_features):
    sns.boxplot(data=df, x=target, y=feat, ax=axes[i], palette='RdYlGn_r')
    axes[i].set_title(feat)
    axes[i].set_xlabel('DR Grade')

# Hide unused subplots
for j in range(len(numeric_features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions by DR Grade', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Correlation Heatmap

In [ ]:
corr_cols = numeric_features + [target]
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 6. Top Correlated Features with DR Grade

In [ ]:
target_corr = corr_matrix[target].drop(target).abs().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
target_corr.plot(kind='barh', color='steelblue')
plt.xlabel('Absolute Correlation with DR Grade')
plt.title('Feature Importance (Correlation-based)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print('Top correlated features:')
print(target_corr)

## 7. Pairwise Feature Relationships

In [ ]:
# Select top 4 features for pairplot
top_features = target_corr.head(4).index.tolist()

g = sns.pairplot(df, vars=top_features, hue=target,
                 palette='RdYlGn_r', diag_kind='kde', height=2.5)
g.figure.suptitle('Pairwise Relationships — Top 4 Features', y=1.02)
plt.show()

## Summary

Key findings from the biomarker EDA:
1. **Class imbalance**: Check if DR grades are balanced — may need oversampling or class weights
2. **Key biomarkers**: HbA1c, diabetes duration, and blood pressure tend to correlate most with DR severity
3. **Missing data**: Identify columns that need imputation before modeling
4. **Scaling**: Numeric features span different ranges — StandardScaler recommended